# TESSERA embeddings — spatial autocorrelation range

**Purpose.** The supervisor raised (2026-11-05 meeting, recalled
2026-11-06) that spatial validation on the expert-polygon side needs a
**buffer** between training and validation points, not just the
coarser 10 km block holdout — otherwise a validation pixel can sit
right next to a training pixel from the same or a neighbouring
polygon. He gave "20/30" as the buffer size, units unconfirmed.

**Established fact (literature, full text read):** the buffer/exclusion
distance used in spatial leave-one-out (SLOO) and buffered
cross-validation is **not a fixed universal constant** — it should be
set to the **range of spatial autocorrelation** of the predictors,
outcome, or model residuals, usually estimated with an empirical
variogram (Le Rest et al., 2014; Mahoney, Johnson & Silge, 2023 — see
[`docs/literature.md#spatial-validation`](../../docs/literature.md#spatial-validation)).
Karasiak et al.'s own software, Museo ToolBox, expresses this buffer
in **pixels**, not metres — which would put "20/30" at 200–300 m at
TESSERA's 10 m resolution, not 20–30 m.

**This notebook estimates that range directly from our own data**,
instead of relying only on the recollection of a verbal "20/30" —
an empirical variogram of the TESSERA embeddings themselves (one of
Le Rest et al.'s three suggested targets: predictors, outcome, or
residuals; this uses the predictors, since no single fitted model
exists yet to take residuals from).


> **Question numbering (note of 2026-11-13).** Current numbering in
> `docs/research_questions.md`: RQ1 = effort-reduction curve (citizen
> science against expert data); RQ2 = from points to reference data
> (segmentation mask, grade and accuracy filters); RQ3 =
> transferability away from the sampled area; RQ4 = TESSERA against
> Sentinel-2.
> In this notebook "RQ1 pilot" is the name of the stored embedding set
> (see `data/README.md`), not a question number.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist
from sklearn.preprocessing import StandardScaler

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from src.background import load_background
EMB_DIR = ROOT / "data" / "processed" / "embeddings"
SEED = 0
N_SAMPLE = 2500


def load_pilot_embeddings():
    px, py, emb = [], [], []
    for b in sorted((EMB_DIR / "tessera_v1.1_2023_rq1_pilot").glob("block_*.npz")):
        d = np.load(b)
        px.append(d["px"]); py.append(d["py"]); emb.append(d["emb"])
    return np.concatenate(px), np.concatenate(py), np.concatenate(emb)


px, py, emb = load_pilot_embeddings()
valid = ~np.isnan(emb).any(axis=1)
px, py, emb = px[valid], py[valid], emb[valid]

# Built in notebooks/05_data_integration/background_pools.ipynb; load_background() checks the
# file and its inputs against data/processed_manifest.csv before loading.
bg = load_background(ROOT, "uniform")
px = np.concatenate([px, bg["x"]])
py = np.concatenate([py, bg["y"]])
emb = np.concatenate([emb, bg["emb"]])
print(f"{len(px):,} points available (RQ1 pilot presences/old COS negatives + new uniform background)")


## Empirical variogram

Semivariance `gamma(h)` = mean squared difference (per embedding
dimension, after standardising) between all point pairs at spatial
lag `h`, binned by distance. Subsampled to 2 500 points (≈3.1M pairs)
to keep the pairwise distance computation tractable; points combine
presences, old COS negatives and the new uniform background, so the
spatial spread covers the whole pilot extent, not just where acacia
is.


In [ ]:
rng = np.random.default_rng(SEED)
idx = rng.choice(len(px), size=min(N_SAMPLE, len(px)), replace=False)
px_s, py_s, emb_s = px[idx], py[idx], emb[idx]

emb_std = StandardScaler().fit_transform(emb_s)
coords = np.column_stack([px_s, py_s])

dist = pdist(coords, metric="euclidean")
gamma = pdist(emb_std, metric="sqeuclidean") / emb_std.shape[1]

bins = np.array([0, 10, 20, 30, 50, 75, 100, 150, 200, 300, 400, 500,
                  750, 1000, 1500, 2000, 3000, 5000, 10000, 20000])
bin_idx = np.digitize(dist, bins)
rows = []
for b in range(1, len(bins)):
    mask = bin_idx == b
    if mask.sum() < 30:
        continue
    rows.append((bins[b - 1], bins[b], int(mask.sum()), gamma[mask].mean()))

variogram = pd.DataFrame(rows, columns=["lag_from_m", "lag_to_m", "n_pairs", "mean_gamma"])
variogram


In [ ]:
import matplotlib.pyplot as plt

mid = (variogram["lag_from_m"] + variogram["lag_to_m"]) / 2
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(mid, variogram["mean_gamma"], marker="o")
ax.set_xscale("log")
ax.set_xlabel("spatial lag (m, log scale)")
ax.set_ylabel("semivariance (gamma)")
ax.set_title("Empirical variogram, TESSERA embeddings (standardised, 128 dims)")
ax.axhline(variogram["mean_gamma"].iloc[-3:].mean(), color="gray", linestyle="--", label="approx. sill")
ax.legend()
plt.show()


## Reading

Semivariance rises steeply from the first bin (0.34 at 10–20 m) to
200–300 m (1.72), barely changes at 300–400 m (1.74), rises again at
400–500 m (2.00) and stays between 2.0 and 2.2 from 500 m onward. Read
off the binned means, it levels off **between 500 m and 1 km**: beyond
that distance two locations' embeddings are no more different, on
average, than any other two random locations. That distance is the
empirical **range of spatial autocorrelation**, the quantity Le Rest
et al. (2014) and the buffered-CV literature say the exclusion buffer
should match.

**The range is above both readings of "20/30".** At 200–300 m
("20–30 pixels" at TESSERA's 10 m resolution) the semivariance is at
about 80% of the level it holds beyond 500 m, and at 20–30 m at about
40%. The variogram therefore does not single out the pixel reading;
either buffer would sit inside the rising part of the curve.

**Caveats (first pass, not definitive):**
- Mixes presences, old COS-era negatives and the new uniform
  background across the whole ~100×65 km pilot extent — density and
  land cover are uneven across that area, so this is a rough,
  area-wide estimate, not a local one.
- Uses the predictors (embeddings) directly, one of three valid
  targets in the literature (Le Rest et al., 2014) — not the outcome
  variable or a fitted model's residuals, since no single model is
  fitted yet across this whole area.
- A proper variogram fit (e.g. exponential/spherical model, confidence
  interval on the range) was not done — this is a binned empirical
  estimate read off a plot, not a fitted range.
- The first two bins hold few pairs (61 and 48).
